# 🧪 Simulação de Teste A/B — Dataset Sintético
**Entendendo o poder estatístico, tamanho de amostra e armadilhas comuns em experimentos controlados**

---

## 🎯 Objetivo

Diferente do notebook anterior (dataset real), aqui **controlamos todos os parâmetros do experimento**.
Isso nos permite responder perguntas fundamentais:

- O que acontece quando aumentamos o tamanho da amostra?
- Como o poder estatístico evolui com mais dados?
- Como simular um falso positivo — e por que isso é perigoso?
- Qual é o tamanho de efeito mínimo que conseguimos detectar?

---

## 0. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)

COR_A = '#1a6b3c'
COR_B = '#f5c518'
COR_DESTAQUE = '#d62728'
COR_INFO = '#2196F3'

os.makedirs('../images_sim', exist_ok=True)
print('✅ Ambiente configurado!')

---
## 1. Cenário Base — Experimento Controlado

Vamos simular um e-commerce testando duas versões de botão de compra.
Definimos que a **Página B tem 15% mais tempo de permanência** que a Página A.

In [ ]:
# Parâmetros do experimento
MEDIA_A = 3.0      # minutos — versão atual
EFEITO  = 0.15     # 15% de melhoria na versão B
MEDIA_B = MEDIA_A * (1 + EFEITO)
DESVIO  = 1.2      # desvio padrão (igual nos dois grupos)
N       = 200      # usuários por grupo
ALPHA   = 0.05

grupo_a = np.random.normal(MEDIA_A, DESVIO, N)
grupo_b = np.random.normal(MEDIA_B, DESVIO, N)
grupo_a = np.clip(grupo_a, 0.1, None)
grupo_b = np.clip(grupo_b, 0.1, None)

stat, p_value = stats.ttest_ind(grupo_a, grupo_b)

print('🔬 Parâmetros do Experimento:')
print(f'   Média real Página A: {MEDIA_A:.2f} min')
print(f'   Média real Página B: {MEDIA_B:.2f} min (+{EFEITO*100:.0f}%)')
print(f'   Desvio padrão: {DESVIO}')
print(f'   N por grupo: {N}')
print()
print('📊 Resultados Observados:')
print(f'   Média observada A: {grupo_a.mean():.2f} min')
print(f'   Média observada B: {grupo_b.mean():.2f} min')
print(f'   P-value: {p_value:.4f}')
print(f'   Resultado: {"✅ Significativo — H₀ rejeitada" if p_value < ALPHA else "❌ Não significativo"}')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(grupo_a, bins=30, color=COR_A, alpha=0.7, label=f'Página A (μ={grupo_a.mean():.2f})', edgecolor='white')
axes[0].hist(grupo_b, bins=30, color=COR_B, alpha=0.7, label=f'Página B (μ={grupo_b.mean():.2f})', edgecolor='white')
axes[0].axvline(grupo_a.mean(), color=COR_A, linestyle='--', linewidth=2)
axes[0].axvline(grupo_b.mean(), color=COR_B, linestyle='--', linewidth=2)
axes[0].set_title(f'Distribuição Simulada\nn={N}/grupo | efeito={EFEITO*100:.0f}%', fontweight='bold')
axes[0].set_xlabel('Tempo (minutos)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

categorias = ['Página A', 'Página B']
medias = [grupo_a.mean(), grupo_b.mean()]
erros = [grupo_a.sem(), grupo_b.sem()]
bars = axes[1].bar(categorias, medias, color=[COR_A, COR_B], edgecolor='white',
                   yerr=erros, capsize=10, error_kw={'linewidth': 2})
for bar, val in zip(bars, medias):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f} min', ha='center', fontweight='bold', fontsize=12)
axes[1].set_title(f'Médias com IC 95%\np-value={p_value:.4f} | {"Significativo ✅" if p_value < ALPHA else "Não Significativo ❌"}', fontweight='bold')
axes[1].set_ylabel('Tempo médio (minutos)')
axes[1].set_ylim(0, max(medias) * 1.4)

plt.tight_layout()
plt.savefig('../images_sim/01_cenario_base.png', dpi=150)
plt.show()

---
## 2. Impacto do Tamanho de Amostra

O que acontece com o p-value quando variamos o número de usuários,
mantendo o **mesmo efeito real de 15%**?

In [ ]:
tamanhos = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 750, 1000]
resultados = []

for n in tamanhos:
    p_values = []
    for _ in range(500):  # 500 simulações por tamanho
        a = np.random.normal(MEDIA_A, DESVIO, n)
        b = np.random.normal(MEDIA_B, DESVIO, n)
        _, p = stats.ttest_ind(a, b)
        p_values.append(p)
    p_mediano = np.median(p_values)
    poder = np.mean(np.array(p_values) < ALPHA)
    resultados.append({'n': n, 'p_mediano': p_mediano, 'poder': poder})

df_res = pd.DataFrame(resultados)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# P-value mediano
axes[0].plot(df_res['n'], df_res['p_mediano'], marker='o', color=COR_A, linewidth=2.5)
axes[0].axhline(ALPHA, color=COR_DESTAQUE, linestyle='--', linewidth=2, label=f'α = {ALPHA}')
axes[0].fill_between(df_res['n'], df_res['p_mediano'], ALPHA,
                     where=(df_res['p_mediano'] > ALPHA), alpha=0.15, color=COR_DESTAQUE, label='Não significativo')
axes[0].fill_between(df_res['n'], df_res['p_mediano'], ALPHA,
                     where=(df_res['p_mediano'] <= ALPHA), alpha=0.15, color=COR_A, label='Significativo')
axes[0].set_title('P-Value Mediano por Tamanho de Amostra\n(efeito real fixo em 15%)', fontweight='bold')
axes[0].set_xlabel('N por grupo')
axes[0].set_ylabel('P-Value')
axes[0].legend()
axes[0].set_xscale('log')

# Poder estatístico
axes[1].plot(df_res['n'], df_res['poder'] * 100, marker='s', color=COR_B, linewidth=2.5)
axes[1].axhline(80, color=COR_DESTAQUE, linestyle='--', linewidth=2, label='Poder mínimo recomendado (80%)')
axes[1].fill_between(df_res['n'], df_res['poder'] * 100, 80,
                     where=(df_res['poder'] * 100 < 80), alpha=0.15, color=COR_DESTAQUE)
axes[1].fill_between(df_res['n'], df_res['poder'] * 100, 80,
                     where=(df_res['poder'] * 100 >= 80), alpha=0.15, color=COR_A)
n_80 = df_res[df_res['poder'] >= 0.80]['n'].min()
axes[1].axvline(n_80, color=COR_A, linestyle=':', linewidth=2, label=f'Mínimo para 80% de poder: n={n_80}')
axes[1].set_title('Poder Estatístico por Tamanho de Amostra\n(efeito real fixo em 15%)', fontweight='bold')
axes[1].set_xlabel('N por grupo')
axes[1].set_ylabel('Poder Estatístico (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('../images_sim/02_impacto_amostra.png', dpi=150)
plt.show()

print(f'💡 Para detectar um efeito de 15% com 80% de poder → precisamos de n={n_80} por grupo ({n_80*2} total)')

---
## 3. Múltiplos Cenários de Efeito

E se o efeito real for diferente? Vamos comparar como o experimento se comporta
para efeitos de 5%, 10%, 15%, 20% e 30%.

In [ ]:
efeitos = [0.05, 0.10, 0.15, 0.20, 0.30]
tamanhos_plot = [10, 20, 50, 100, 200, 500, 1000]
cores_efeito = ['#d62728', '#ff7f0e', '#2196F3', '#9467bd', '#1a6b3c']

fig, ax = plt.subplots(figsize=(14, 6))

for efeito, cor in zip(efeitos, cores_efeito):
    poderes = []
    for n in tamanhos_plot:
        p_vals = []
        for _ in range(300):
            a = np.random.normal(MEDIA_A, DESVIO, n)
            b = np.random.normal(MEDIA_A * (1 + efeito), DESVIO, n)
            _, p = stats.ttest_ind(a, b)
            p_vals.append(p)
        poderes.append(np.mean(np.array(p_vals) < ALPHA) * 100)
    ax.plot(tamanhos_plot, poderes, marker='o', color=cor, linewidth=2.5,
            label=f'Efeito {int(efeito*100)}%')

ax.axhline(80, color='black', linestyle='--', linewidth=1.5, label='Poder mínimo (80%)')
ax.set_title('Poder Estatístico por Tamanho de Efeito e Amostra\nQual efeito consigo detectar com minha amostra?',
             fontsize=13, fontweight='bold')
ax.set_xlabel('N por grupo')
ax.set_ylabel('Poder Estatístico (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xscale('log')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../images_sim/03_cenarios_efeito.png', dpi=150)
plt.show()

---
## 4. ⚠️ Armadilha: O Falso Positivo (Erro Tipo I)

O que acontece quando **não há diferença real** entre A e B,
mas rodamos o teste várias vezes ou com múltiplas métricas?

Esta é uma das armadilhas mais comuns em experimentos corporativos.

In [ ]:
# Simulando 1000 experimentos onde NÃO há diferença real
n_experimentos = 1000
n_usuarios = 100
falsos_positivos = []
todos_pvalues = []

for _ in range(n_experimentos):
    # Ambos os grupos com MESMA média — não há efeito real
    a = np.random.normal(MEDIA_A, DESVIO, n_usuarios)
    b = np.random.normal(MEDIA_A, DESVIO, n_usuarios)  # mesma média!
    _, p = stats.ttest_ind(a, b)
    todos_pvalues.append(p)
    if p < ALPHA:
        falsos_positivos.append(p)

taxa_fp = len(falsos_positivos) / n_experimentos * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de p-values
axes[0].hist(todos_pvalues, bins=30, color=COR_INFO, edgecolor='white', alpha=0.8)
axes[0].axvline(ALPHA, color=COR_DESTAQUE, linestyle='--', linewidth=2.5, label=f'α = {ALPHA}')
axes[0].fill_between([0, ALPHA], 0, 50, alpha=0.2, color=COR_DESTAQUE, label=f'Falsos positivos ({taxa_fp:.1f}%)')
axes[0].set_title(f'Distribuição de P-Values\n({n_experimentos} experimentos SEM diferença real)', fontweight='bold')
axes[0].set_xlabel('P-Value')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Acúmulo de falsos positivos com múltiplos testes
n_testes = range(1, 21)
prob_pelo_menos_um_fp = [1 - (1 - ALPHA) ** n for n in n_testes]
axes[1].plot(n_testes, [p * 100 for p in prob_pelo_menos_um_fp],
             marker='o', color=COR_DESTAQUE, linewidth=2.5)
axes[1].axhline(5, color='gray', linestyle=':', linewidth=1.5, label='α = 5% (teste único)')
axes[1].axhline(50, color=COR_DESTAQUE, linestyle='--', linewidth=1.5, label='50% de chance de FP')
axes[1].set_title('Probabilidade de Falso Positivo\nconforme aumentamos o número de testes',
                  fontweight='bold')
axes[1].set_xlabel('Número de métricas/testes simultâneos')
axes[1].set_ylabel('Probabilidade de pelo menos 1 FP (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.suptitle('⚠️ A Armadilha do Falso Positivo em Testes A/B', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images_sim/04_falso_positivo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'⚠️  Em {n_experimentos} experimentos sem diferença real:')
print(f'   Falsos positivos: {len(falsos_positivos)} ({taxa_fp:.1f}%)')
print(f'   Isso é exatamente o esperado com α = {ALPHA} ({ALPHA*100:.0f}%)')
print(f'\n💡 Com 14 métricas testadas simultaneamente, há 50% de chance de um falso positivo!')
print(f'   Correção de Bonferroni: use α/{14} = {ALPHA/14:.4f} para cada teste individual')

---
## 5. ⚠️ Armadilha: Parar o Teste Cedo (Peeking Problem)

In [ ]:
# Simulando o que acontece quando checamos o p-value a cada novo usuário
N_MAX = 500
n_simulacoes = 200
parou_cedo = 0
p_ao_longo_do_tempo = []

# Uma simulação exemplo
dados_a = np.random.normal(MEDIA_A, DESVIO, N_MAX)
dados_b = np.random.normal(MEDIA_A, DESVIO, N_MAX)  # sem efeito real
p_trajetoria = []

for i in range(10, N_MAX + 1):
    _, p = stats.ttest_ind(dados_a[:i], dados_b[:i])
    p_trajetoria.append((i, p))

df_traj = pd.DataFrame(p_trajetoria, columns=['n', 'p_value'])

# Múltiplas simulações
for _ in range(n_simulacoes):
    a = np.random.normal(MEDIA_A, DESVIO, N_MAX)
    b = np.random.normal(MEDIA_A, DESVIO, N_MAX)
    for i in range(10, N_MAX + 1, 10):
        _, p = stats.ttest_ind(a[:i], b[:i])
        if p < ALPHA:
            parou_cedo += 1
            break

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_traj['n'], df_traj['p_value'], color=COR_A, linewidth=1.5, alpha=0.8)
ax.axhline(ALPHA, color=COR_DESTAQUE, linestyle='--', linewidth=2, label=f'α = {ALPHA}')
ax.fill_between(df_traj['n'], df_traj['p_value'], ALPHA,
                where=(df_traj['p_value'] < ALPHA), alpha=0.3, color=COR_DESTAQUE,
                label='"Significativo" — mas não é!')

primeiros_cruzamentos = df_traj[df_traj['p_value'] < ALPHA]
if not primeiros_cruzamentos.empty:
    primeiro = primeiros_cruzamentos.iloc[0]
    ax.annotate(f'Se parasse aqui (n={int(primeiro["n"])})\np={primeiro["p_value"]:.3f} — falso positivo!',
                xy=(primeiro['n'], primeiro['p_value']),
                xytext=(primeiro['n'] + 50, primeiro['p_value'] + 0.15),
                arrowprops=dict(arrowstyle='->', color=COR_DESTAQUE),
                fontsize=10, color=COR_DESTAQUE, fontweight='bold')

ax.set_title('O Problema de Checar o P-Value Continuamente\n("Peeking Problem") — sem efeito real nos dados',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Número de usuários no experimento')
ax.set_ylabel('P-Value')
ax.legend()
plt.tight_layout()
plt.savefig('../images_sim/05_peeking_problem.png', dpi=150)
plt.show()

print(f'⚠️  Em {n_simulacoes} simulações sem efeito real, parar cedo levou a')
print(f'   falsos positivos em {parou_cedo} casos ({parou_cedo/n_simulacoes*100:.1f}%)')
print(f'\n💡 Solução: defina o tamanho de amostra ANTES de iniciar o experimento')
print(f'   e só olhe o resultado quando atingir esse número.')

---
## 6. Dashboard Final — Guia de Decisão para Testes A/B

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Quadrante 1: Tamanho de amostra vs efeito
efeitos_grid = np.arange(0.05, 0.55, 0.05)
from scipy.stats import norm
def n_minimo(efeito, desvio=DESVIO, media=MEDIA_A, alpha=0.05, poder=0.80):
    z_a = norm.ppf(1 - alpha/2)
    z_b = norm.ppf(poder)
    delta = media * efeito
    return int(np.ceil(((z_a + z_b) * desvio / delta) ** 2))

ns = [n_minimo(e) for e in efeitos_grid]
axes[0, 0].bar([f'{int(e*100)}%' for e in efeitos_grid], ns, color=COR_A, edgecolor='white')
axes[0, 0].set_title('Amostra Mínima por Efeito\n(α=0.05, poder=80%)', fontweight='bold')
axes[0, 0].set_xlabel('Efeito mínimo detectável')
axes[0, 0].set_ylabel('N por grupo')
axes[0, 0].tick_params(axis='x', rotation=45)

# Quadrante 2: Distribuição de erros
tipos = ['Verdadeiro\nPositivo', 'Falso\nPositivo\n(Tipo I)', 'Falso\nNegativo\n(Tipo II)', 'Verdadeiro\nNegativo']
valores = [75, 5, 15, 80]
cores_q = [COR_A, COR_DESTAQUE, '#ff7f0e', COR_INFO]
bars = axes[0, 1].bar(tipos, valores, color=cores_q, edgecolor='white')
for bar, val in zip(bars, valores):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val}%', ha='center', fontweight='bold')
axes[0, 1].set_title('Tipos de Erro em Testes A/B\n(referência teórica com α=0.05, poder=80%)', fontweight='bold')
axes[0, 1].set_ylabel('Probabilidade (%)')

# Quadrante 3: Evolução do poder com mais simulações
ns_evolucao = [50, 100, 200, 300, 500]
poderes_evolucao = []
for n in ns_evolucao:
    p_vals = []
    for _ in range(500):
        a = np.random.normal(MEDIA_A, DESVIO, n)
        b = np.random.normal(MEDIA_B, DESVIO, n)
        _, p = stats.ttest_ind(a, b)
        p_vals.append(p)
    poderes_evolucao.append(np.mean(np.array(p_vals) < ALPHA) * 100)

axes[1, 0].plot(ns_evolucao, poderes_evolucao, marker='o', color=COR_B, linewidth=2.5)
axes[1, 0].axhline(80, color=COR_DESTAQUE, linestyle='--', linewidth=2, label='80% (recomendado)')
axes[1, 0].fill_between(ns_evolucao, poderes_evolucao, 80,
                        where=[p >= 80 for p in poderes_evolucao],
                        alpha=0.2, color=COR_A)
axes[1, 0].set_title('Evolução do Poder Estatístico\n(efeito real = 15%)', fontweight='bold')
axes[1, 0].set_xlabel('N por grupo')
axes[1, 0].set_ylabel('Poder (%)')
axes[1, 0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1, 0].legend()

# Quadrante 4: Checklist de boas práticas
axes[1, 1].axis('off')
checklist = [
    '✅ Definir H₀ e H₁ antes de coletar dados',
    '✅ Calcular tamanho de amostra antes do experimento',
    '✅ Verificar normalidade antes do teste',
    '✅ Usar α = 0.05 e poder ≥ 80%',
    '✅ Não parar o teste antes do n planejado',
    '✅ Aplicar correção de Bonferroni para múltiplos testes',
    '❌ Checar p-value durante o experimento',
    '❌ Escolher a métrica após ver os resultados',
    '❌ Confundir significância com relevância prática',
]
axes[1, 1].text(0.05, 0.95, '📋 Boas Práticas em Teste A/B',
               transform=axes[1, 1].transAxes, fontsize=13,
               fontweight='bold', va='top')
for i, item in enumerate(checklist):
    cor = COR_A if item.startswith('✅') else COR_DESTAQUE
    axes[1, 1].text(0.05, 0.82 - i * 0.09, item,
                   transform=axes[1, 1].transAxes, fontsize=10,
                   color=cor, va='top')

plt.suptitle('Dashboard — Guia Completo de Teste A/B', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../images_sim/06_dashboard_final.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 Conclusões

| Conceito | Lição Aprendida |
|----------|----------------|
| **Tamanho de amostra** | Quanto menor o efeito real, mais usuários precisamos |
| **Poder estatístico** | Experimentos com baixo poder perdem efeitos reais |
| **Falso positivo** | Com α=0.05, 5% dos experimentos darão significativos por acaso |
| **Múltiplos testes** | Testar 14 métricas = 50% de chance de ao menos um falso positivo |
| **Peeking problem** | Checar o resultado antes da amostra planejada infla os falsos positivos |
| **Boas práticas** | Planejar o experimento antes de coletá-lo é fundamental |

---
**Próximo passo:** Aplicar Teste A/B em dados reais de conversão com API de analytics (Google Analytics / Mixpanel).